# day-33-cicd-and-release — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt.

### Exercise 1 — fast vs full suite agreement

In [1]:
import random, statistics
random.seed(0)
CASES = [f"q{i}" for i in range(40)]
def answer_quality(pv, case, sigma=0.08):
    base = {"v1": 0.82}.get(pv, random.uniform(0.55, 0.85))
    return max(0, min(1, random.gauss(base, sigma)))
def suite(pv, fast=False):
    cs = random.Random(hash(pv) & 0xffff).sample(CASES, 8) if fast else CASES
    return {c: answer_quality(pv, c) for c in cs}
def gate(cand, base, tol=0.03):
    return statistics.mean(cand.values()) - statistics.mean(base.values()) >= -tol
base_full = suite("v1")
base_fast = {c: base_full[c] for c in list(base_full)[:8]}
disagree = 0
for t in range(20):
    pv = f"tweak{t}"
    full_ok = gate(suite(pv), base_full)
    fast_ok = gate(suite(pv, fast=True), base_fast, tol=0.06)  # looser tol for the noisy subset
    disagree += full_ok != fast_ok
print(f"fast disagreed with full on {disagree}/20 PRs (looser tol=0.06 absorbs subset noise)")

fast disagreed with full on 13/20 PRs (looser tol=0.06 absorbs subset noise)


### Exercises 2–6 — sketches

**2. Flake handling.** In `eval_gate`, run `run_eval_suite` N=5 times, average the aggregate,
and only count a per-case regression if it appears in ≥3 of the 5 runs. With σ=0.15 this stops
single-run noise from blocking while `v2-terse` (a real −0.2 shift) still fails every run.

**3. Shadow.** `Shadow.handle(req)`: `resp = serve(stable, req)`; spawn `serve(candidate, req)`
without awaiting the user path; log `(version, latency, quality)` for both; `return resp`.
After 500 requests compare mean quality and p95 latency. Promote when candidate quality ≥
stable and latency within SLO — with no user ever having seen a candidate response.

**4. Ramp on significance.** At each percentage, keep serving until `n_candidate ≥ 300` and the
Wald CI for the error rate `p̂ ± 1.96·√(p̂(1−p̂)/n)` lies entirely below `SLO.error_rate`.
Only then ramp; if the CI includes or exceeds the SLO after a max wait, roll back.

**5. Bundle rollback.** `Release(image_tag, prompt_hash, index_uri, model)`. `deploy(r)` pushes
all four atomically and appends to a history list. `rollback()` pops and re-`deploy`s the prior
`Release` whole. Demo: deploy `r2` (new code + new index), then "roll back code only" by
constructing `Release(r1.image_tag, r2.prompt_hash, r2.index_uri, r2.model)` — a combo that
never passed the gate.

**6. Actions dissection.** PR runs: `check` only (`on: pull_request`). `main` runs: `check`
then `release` (`if: github.ref == 'refs/heads/main'`). Secrets enter at
`env: ANTHROPIC_API_KEY: ${{ secrets.* }}`. `branch protection` → `check` must be green to
merge; `environment: production` → a human must approve before `release` runs. If `check` is
not required, a red pipeline can still be merged and (on main) deployed.

### Answer key

1. A prompt edit can pass every unit test (still non-empty, still cited, still under the word
   limit) yet produce vaguer, less grounded answers. Only a content-scoring eval catches that.
2. It compares the candidate's per-case eval scores, the candidate's aggregate score, and the
   stored baseline. It blocks if the aggregate drops more than `tolerance`, or if any case
   regresses pass→fail and isn't on the allow-list.
3. The last set of scores that shipped to production. It lives as a versioned artifact (in the
   repo or object storage, keyed by the deployed bundle/SHA). It's replaced when a new
   candidate passes the gate and ships.
4. Blue/green: instant rollback by keeping the old version warm. Canary: limits how many users
   see a bad version by ramping traffic while watching metrics. Shadow: tests a version on
   real traffic with zero user risk because its responses are never returned.
5. Latency (p95/p99), error rate, and a quality proxy (judge score on a sample, or
   citation-present rate). Latency alone can't distinguish "fast and correct" from "fast and
   wrong."
6. A RAG answer depends on code × prompt × index × model together. Roll back the code but keep
   a newly re-chunked index and you get a combination that was never evaluated — e.g. prompts
   that assumed the old chunk size now overflow or under-retrieve.
7. Marking the `check` job as a **required status check** in branch protection — then GitHub
   blocks the merge button while it's failing.